pip install -U langgraph

In [ ]:
from langgraph.graph import StateGraph
from typing_extensions import TypedDict

#定义输入格式
class InputState(TypedDict):
    question: str

#定义输出格式
class OutputState(TypedDict):
    answer: str

# 将InputState和OutputState 这两个TypedDict 类型合并成一个字典类型
class OverallState(InputState, OutputState):
    pass


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # 加载.env文件里的变量
print(os.getenv("DEEPSEEK_API_KEY"))  # 现在可以正常读取了

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

import getpass


def llm_node(state:InputState):
    messages = [
        {"role":"system","content":"你是一位乐于助人的智能小助理"},
        {"role":"user","content":state["question"]}
    ]

    llm = ChatOpenAI(
        model="deepseek-chat",  # 使用的模型名称，目前官方推荐用 'deepseek-chat'
        api_key=os.getenv("DEEPSEEK_API_KEY"),  # 你的 DeepSeek API Key
        base_url="https://api.deepseek.com/v1",  # DeepSeek API 地址
        temperature=0,
    )

    response=llm.invoke(messages)
    return {"answer":response.content}

In [ ]:
from langgraph.graph import START, END
builder= StateGraph(OverallState, input=InputState,output=OutputState)
builder.add_node("llm_node",llm_node)

builder.add_edge(START,"llm_node")
builder.add_edge("llm_node",END)

graph=builder.compile()

In [ ]:
graph.invoke({"question":"你好！"})

In [ ]:
final_answer=graph.invoke({"question":"你好,请详细介绍一下你自己"})
print(final_answer["answer"])